In [1]:
import os
import numpy as np
import pandas as pd
import xarray as xr

In [2]:
ssrd_path = "../.data/ERA5/era5_ssrd_dailymean_JJA_NE_2000_2025_masked.nc"
d2m_path  = "../.data/ERA5/era5_d2m_dailymean_JJA_NE_2000_2025_masked.nc"
base_path = "../.data/downscaling_splits/train.nc"  # only used to borrow valid_mask/grid consistency if needed

ds_ssrd = xr.open_dataset(ssrd_path)
ds_d2m = xr.open_dataset(d2m_path)

In [3]:
def standardize_time_dim(ds):
    rename_dict = {}

    if "day" in ds.dims:
        rename_dict["day"] = "time"
    if "day" in ds.coords:
        rename_dict["day"] = "time"

    if "datetime" in ds.dims:
        rename_dict["datetime"] = "time"
    if "datetime" in ds.coords:
        rename_dict["datetime"] = "time"

    if rename_dict:
        ds = ds.rename(rename_dict)

    return ds


def get_main_var_name(ds):
    var_names = list(ds.data_vars)
    if len(var_names) != 1:
        raise ValueError(f"Expected exactly 1 data variable, found {var_names}")
    return var_names[0]


def rename_coarse_grid(da):
    rename_dict = {}
    if "lat" in da.dims:
        rename_dict["lat"] = "lat_coarse"
    if "lon" in da.dims:
        rename_dict["lon"] = "lon_coarse"
    if rename_dict:
        da = da.rename(rename_dict)
    return da


def subset_years(da, start_year, end_year):
    years = da["time"].dt.year
    return da.sel(time=(years >= start_year) & (years <= end_year))

In [4]:
ds_ssrd = standardize_time_dim(ds_ssrd)
ds_d2m = standardize_time_dim(ds_d2m)

ssrd_name = get_main_var_name(ds_ssrd)
d2m_name = get_main_var_name(ds_d2m)

print("SSRD variable:", ssrd_name)
print("D2M variable :", d2m_name)

ssrd = ds_ssrd[ssrd_name].rename("ssrd_lowres")
d2m = ds_d2m[d2m_name].rename("d2m_lowres")

SSRD variable: ssrd
D2M variable : d2m


In [5]:
common_time = np.intersect1d(
    pd.to_datetime(ssrd["time"].values),
    pd.to_datetime(d2m["time"].values)
)

print("Common days:", len(common_time))

ssrd = ssrd.sel(time=common_time)
d2m = d2m.sel(time=common_time)

Common days: 2392


In [6]:
ssrd = rename_coarse_grid(ssrd)
d2m = rename_coarse_grid(d2m)

print(ssrd.shape, d2m.shape)

(2392, 39, 51) (2392, 39, 51)


In [7]:
ds_train_base = xr.open_dataset("../.data/downscaling_splits/train.nc")
valid_mask = ds_train_base["valid_mask"].astype("int8")
valid_mask

<xarray.DataArray 'valid_mask' (lat: 240, lon: 311)> Size: 75kB
array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]], dtype=int8)
Coordinates:
  * lat      (lat) float64 2kB 46.98 46.94 46.9 46.86 ... 37.11 37.07 37.03
  * lon      (lon) float64 2kB -79.97 -79.93 -79.89 ... -67.14 -67.1 -67.06
Attributes:
    units:              K
    description:        Daily Maximum Temperature
    long_name:          tmmx
    standard_name:      tmmx
    dimensions:         lon lat time
    grid_mapping:       crs
    coordinate_system:  WGS84,EPSG:4326

In [8]:
splits = {
    "train": (2000, 2018),
    "val":   (2019, 2021),
    "test":  (2022, 2025),
}

out_dir = "../.data/downscaling_splits_extra"
os.makedirs(out_dir, exist_ok=True)

In [9]:
for split_name, (start_year, end_year) in splits.items():
    ssrd_split = subset_years(ssrd, start_year, end_year)
    d2m_split = subset_years(d2m, start_year, end_year)

    if ssrd_split.sizes["time"] != d2m_split.sizes["time"]:
        raise ValueError(
            f"{split_name}: time length mismatch: "
            f"ssrd={ssrd_split.sizes['time']}, d2m={d2m_split.sizes['time']}"
        )

    ds_out = xr.Dataset(
        data_vars={
            "ssrd_lowres": ssrd_split.astype("float32"),
            "d2m_lowres": d2m_split.astype("float32"),
            "valid_mask": valid_mask.astype("int8"),
        },
        attrs={
            "description": f"{split_name} split for extra ERA5 features",
            "train_period": "2000-2018",
            "val_period": "2019-2021",
            "test_period": "2022-2025",
            "note": "Extra low-resolution ERA5 predictors on native coarse grid. "
                    "Saved separately to avoid modifying baseline Tmax input files."
        }
    )

    encoding = {
        "ssrd_lowres": {"zlib": True, "complevel": 4, "dtype": "float32"},
        "d2m_lowres": {"zlib": True, "complevel": 4, "dtype": "float32"},
        "valid_mask": {"zlib": True, "complevel": 4, "dtype": "int8"},
    }

    out_path = os.path.join(out_dir, f"{split_name}_extra.nc")
    ds_out.to_netcdf(out_path, engine="netcdf4", encoding=encoding)

    t0 = pd.to_datetime(ds_out.time.values[0]).strftime("%Y-%m-%d")
    t1 = pd.to_datetime(ds_out.time.values[-1]).strftime("%Y-%m-%d")

    print(f"Saved {split_name}: {out_path}")
    print(f"  time range: {t0} to {t1}")
    print(f"  n_time = {ds_out.sizes['time']}")

Saved train: ../.data/downscaling_splits_extra/train_extra.nc
  time range: 2000-06-01 to 2018-08-31
  n_time = 1748
Saved val: ../.data/downscaling_splits_extra/val_extra.nc
  time range: 2019-06-01 to 2021-08-31
  n_time = 276
Saved test: ../.data/downscaling_splits_extra/test_extra.nc
  time range: 2022-06-01 to 2025-08-31
  n_time = 368


In [10]:
ds_train_extra = xr.open_dataset("../.data/downscaling_splits_extra/train_extra.nc")
ds_train_extra

<xarray.Dataset> Size: 28MB
Dimensions:      (time: 1748, lat_coarse: 39, lon_coarse: 51, lat: 240, lon: 311)
Coordinates:
  * time         (time) datetime64[ns] 14kB 2000-06-01 2000-06-02 ... 2018-08-31
  * lat_coarse   (lat_coarse) float64 312B 46.75 46.5 46.25 ... 37.75 37.5 37.25
  * lon_coarse   (lon_coarse) float64 408B -79.75 -79.5 -79.25 ... -67.5 -67.25
  * lat          (lat) float64 2kB 46.98 46.94 46.9 46.86 ... 37.11 37.07 37.03
  * lon          (lon) float64 2kB -79.97 -79.93 -79.89 ... -67.14 -67.1 -67.06
Data variables:
    ssrd_lowres  (time, lat_coarse, lon_coarse) float32 14MB ...
    d2m_lowres   (time, lat_coarse, lon_coarse) float32 14MB ...
    valid_mask   (lat, lon) int8 75kB ...
Attributes:
    description:   train split for extra ERA5 features
    train_period:  2000-2018
    val_period:    2019-2021
    test_period:   2022-2025
    note:          Extra low-resolution ERA5 predictors on native coarse grid...